In [1]:
# --- Portable bootstrap: locate scripts/ from any working directory ---
import sys
from pathlib import Path

def _find_scripts_dir():
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        for cand in (base / "scripts", base,
                     *sorted(base.glob("*/scripts")), *sorted(base.glob("*/*/scripts"))):
            if (cand / "project_paths.py").is_file():
                return cand
    raise FileNotFoundError(
        "scripts/project_paths.py not found. Open this notebook from inside the "
        "cloned repository, or point the kernel's working directory at it."
    )

sys.path.insert(0, str(_find_scripts_dir()))
from project_config import *   # SEED, DATA_DIR, ACOUSTIC_FULL, collapse_functionals, repeated_cv, ...

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

# --- Data ---
model_df = load_model_df()
y = model_df["phq9"]
acoustic = ACOUSTIC_FULL            # the 88 raw eGeMAPS features these cells start from

# --- Full-sample correlation structure, for the DENDROGRAM and THRESHOLD-SWEEP
# figures only. Descriptive, never scored -- the modelling cells refit the
# clustering per fold via CorrelationClusterSelector.
_feat = pd.read_csv(EGEMAPS_CSV, dtype={"subject_id": str}).dropna(subset=["phq9"])
corr = np.nan_to_num(spearmanr(_feat[acoustic]).correlation, nan=0.0)
corr = (corr + corr.T) / 2
np.fill_diagonal(corr, 1.0)
Z = linkage(squareform(1.0 - np.abs(corr), checks=False), method="average")


In [2]:
# ===== Correlation clustering as a Pipeline step (refit per training fold) =====
# Was: clustered once on all 52 subjects, then cross-validated the survivors, so every
# fold was scored on features chosen with help from its own test rows. Now it refits per fold.
import numpy as np
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

CORR_THRESH = 0.75

class CorrelationClusterSelector(BaseEstimator, TransformerMixin):
    """Keep one medoid per |Spearman rho| cluster. Fit on training rows only."""
    def __init__(self, threshold=0.75):
        self.threshold = threshold

    def fit(self, X, y=None):
        corr = np.nan_to_num(spearmanr(np.asarray(X, float)).correlation, nan=0.0)
        corr = (corr + corr.T) / 2
        np.fill_diagonal(corr, 1.0)
        cl = fcluster(linkage(squareform(1 - np.abs(corr), checks=False), "average"),
                      t=1 - self.threshold, criterion="distance")
        absc = np.abs(corr)
        self.support_ = np.sort([i[np.argmax(absc[np.ix_(i, i)].sum(1))]
                                 for i in (np.where(cl == c)[0] for c in np.unique(cl))])
        return self

    def transform(self, X):
        return np.asarray(X, float)[:, self.support_]


X, yk = model_df[ACOUSTIC_FULL], model_df["phq9"]
pipe = Pipeline([("cluster", CorrelationClusterSelector(CORR_THRESH)),
                 ("rf",      RandomForestRegressor(random_state=SEED))])

pred = cross_val_predict(pipe, X, yk, cv=KFold(10, shuffle=True, random_state=SEED), n_jobs=1)
print(f"Refit per fold: R2 {r2_score(yk, pred):.3f}  MAE {mean_absolute_error(yk, pred):.3f}")


Refit per fold: R2 0.072  MAE 7.176


In [4]:
from scipy.cluster.hierarchy import fcluster
print(f"{'threshold':>10}{'n_clusters (features kept)':>28}")
for thr in [0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55, 0.5]:
    n = len(np.unique(fcluster(Z, t=1.0 - thr, criterion="distance")))
    print(f"{thr:>10.2f}{n:>28}")


 threshold  n_clusters (features kept)
      0.90                          70
      0.85                          67
      0.80                          57
      0.75                          54
      0.70                          49
      0.65                          42
      0.60                          39
      0.55                          34
      0.50                          31


In [5]:
pipe = Pipeline([("cluster", CorrelationClusterSelector(CORR_THRESH)),
                 ("rf",      RandomForestRegressor(random_state=SEED))])
r = repeated_cv(pipe, model_df[ACOUSTIC_FULL], y)
print(f"Clustered (refit per fold):  R2 {r['R2'][0]:.3f}±{r['R2'][1]:.3f}  MAE {r['MAE'][0]:.3f}")


Clustered (refit per fold):  R2 0.089±0.039  MAE 7.143


In [7]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

y = model_df["phq9"]

def score(est, X, y, seeds=range(20)):
    r2, mae = [], []
    for s in seeds:
        kf = KFold(n_splits=10, shuffle=True, random_state=s)
        p = cross_val_predict(est, X, y, cv=kf, n_jobs=-1)
        r2.append(r2_score(y, p)); mae.append(mean_absolute_error(y, p))
    return np.mean(r2), np.std(r2), np.mean(mae)

def pca_pipe(n):
    return Pipeline([("scale", StandardScaler()),
                     ("pca",   PCA(n_components=n, random_state=SEED)),
                     ("rf",    RandomForestRegressor(random_state=SEED))])

print(f"{'model':16s}{'R2 (mean±sd)':>20}{'MAE':>10}")
r2m, r2s, maem = score(RandomForestRegressor(random_state=SEED), model_df[acoustic], y)
print(f"{'Full 88':16s}{f'{r2m:.3f}±{r2s:.3f}':>20}{maem:>10.3f}")
for n in [0.90, 15, 10, 5]:
    r2m, r2s, maem = score(pca_pipe(n), model_df[acoustic], y)
    label = f"PCA {int(n*100)}% var" if n < 1 else f"PCA {n} comps"
    print(f"{label:16s}{f'{r2m:.3f}±{r2s:.3f}':>20}{maem:>10.3f}")


model                   R2 (mean±sd)       MAE
Full 88                  0.205±0.028     6.414
PCA 90% var             -0.114±0.086     7.734
PCA 15 comps            -0.115±0.075     7.760
PCA 10 comps            -0.123±0.068     7.696
PCA 5 comps             -0.148±0.107     7.726


In [8]:
# A-priori "collapse the functionals": keep mean + one variability measure per LLD,
# drop percentiles, percentile-ranges, and rising/falling-slope functionals.
DROP_MARKERS = ["percentile", "pctlrange", "RisingSlope", "FallingSlope"]
acoustic_collapsed = [c for c in acoustic if not any(m in c for m in DROP_MARKERS)]
dropped = [c for c in acoustic if c not in acoustic_collapsed]

print(f"Collapsed {len(acoustic)} -> {len(acoustic_collapsed)} features (dropped {len(dropped)})")
print("\nDropped functionals:")
for d in dropped: print("  ", d)

r2m, r2s, maem = score(RandomForestRegressor(random_state=SEED), model_df[acoustic_collapsed], y)
print(f"\nCollapsed {len(acoustic_collapsed)}: R2 {r2m:.3f}±{r2s:.3f}  MAE {maem:.3f}")
print("Full 88 baseline:   R2 0.205±0.028  MAE 6.414")


Collapsed 88 -> 72 features (dropped 16)

Dropped functionals:
   F0semitoneFrom27.5Hz_sma3nz_percentile20.0
   F0semitoneFrom27.5Hz_sma3nz_percentile50.0
   F0semitoneFrom27.5Hz_sma3nz_percentile80.0
   F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2
   F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope
   F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope
   F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope
   F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope
   loudness_sma3_percentile20.0
   loudness_sma3_percentile50.0
   loudness_sma3_percentile80.0
   loudness_sma3_pctlrange0-2
   loudness_sma3_meanRisingSlope
   loudness_sma3_stddevRisingSlope
   loudness_sma3_meanFallingSlope
   loudness_sma3_stddevFallingSlope

Collapsed 72: R2 0.228±0.031  MAE 6.282
Full 88 baseline:   R2 0.205±0.028  MAE 6.414


In [9]:
r2m, r2s, maem= score(GradientBoostingRegressor(random_state=SEED), model_df[acoustic_collapsed], y)
print(f"\nCollapsed GBM {len(acoustic_collapsed)}: R2 {r2m:.3f}±{r2s:.3f}  MAE {maem:.3f}")


Collapsed GBM 72: R2 0.200±0.065  MAE 5.978


In [10]:

pd.Series(acoustic_collapsed, name="feature").to_csv(DATA_DIR / "acoustic_collapsed_features.csv", index=False)


In [11]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import pearsonr

# 72-feature acoustic set (functional collapse), single source of truth
acoustic = pd.read_csv(DATA_DIR / "acoustic_collapsed_features.csv")["feature"].tolist()
demo     = ["age", "gender", "education_years"]
psych    = ["ctq_sf", "LES", "SSRS", "gad7", "PSQI"]
clinical = demo + psych
y = model_df["phq9"]

feature_sets = {
    "Acoustic (72)":            acoustic,
    "Demographics + Acoustic":  demo + acoustic,
    "Clinical + Acoustic":      clinical + acoustic,
}
models = {"RF": RandomForestRegressor(random_state=SEED),
          "GBM": GradientBoostingRegressor(random_state=SEED)}

def repeated_cv(est, X, y, seeds=range(20)):
    rows = []
    for s in seeds:
        kf = KFold(n_splits=10, shuffle=True, random_state=s)
        p = cross_val_predict(est, X, y, cv=kf, n_jobs=-1)
        rows.append({"MAE": mean_absolute_error(y, p), "RMSE": np.sqrt(mean_squared_error(y, p)),
                     "R2": r2_score(y, p), "r": pearsonr(y, p)[0]})
    d = pd.DataFrame(rows)
    return {k: (d[k].mean(), d[k].std()) for k in ["MAE", "RMSE", "R2", "r"]}

print(f"{'feature set':26s}{'model':5s}{'R2':>15}{'MAE':>13}{'RMSE':>13}{'r':>13}")
for sname, cols in feature_sets.items():
    for mname, m in models.items():
        r = repeated_cv(m, model_df[cols], y)
        f = lambda k: f"{r[k][0]:.3f}±{r[k][1]:.3f}"
        print(f"{sname:26s}{mname:5s}{f('R2'):>15}{f('MAE'):>13}{f('RMSE'):>13}{f('r'):>13}")


feature set               model             R2          MAE         RMSE            r
Acoustic (72)             RF       0.228±0.032  6.282±0.150  7.380±0.152  0.478±0.032
Acoustic (72)             GBM      0.200±0.067  5.978±0.265  7.505±0.313  0.495±0.048
Demographics + Acoustic   RF       0.244±0.032  6.227±0.126  7.302±0.154  0.496±0.031
Demographics + Acoustic   GBM      0.170±0.080  6.132±0.281  7.646±0.357  0.471±0.052
Clinical + Acoustic       RF       0.762±0.027  2.828±0.157  4.095±0.226  0.875±0.015
Clinical + Acoustic       GBM      0.716±0.039  3.149±0.214  4.463±0.305  0.850±0.022


In [14]:
# ===== Nested CV: unbiased estimate of the SELECTED acoustic reduction pipeline =====
#
# Outer 5-fold: outer_test is never touched during selection.
#   Inner (outer_train only): repeated 10-fold CV over a GRID of (strategy,
#   hyperparameter) configs; the winner is chosen on outer_train alone, refit on
#   outer_train, and scored on the untouched outer_test.
#
# Hyperparameters are NOT fixed from full-sample inspection. The clustering
# threshold and the PCA variance target are searched inside the inner loop, so
# each outer fold picks its own. The GRID itself is an a-priori search space, not
# a tuned value. The selection metric is also not a free choice: the whole
# procedure is run under both MAE and R2 and both outer estimates are reported.
import numpy as np, pandas as pd
from scipy.stats import spearmanr, pearsonr
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# --- Config -----------------------------------------------------------------
N_OUTER     = 5
INNER_SEEDS = range(5)          # repeated 10-fold CV inside each outer fold
N_JOBS      = -1
SELECTION_METRICS = ("MAE", "R2")
HIGHER_IS_BETTER  = {"R2": True, "r": True, "MAE": False, "RMSE": False}

# Search space, fixed a priori: spans the plausible range, not tuned on results
GRID = {
    "A-priori collapse": [{}],                                            # no hyperparameter
    "Corr. clustering":  [{"threshold": t} for t in (0.60, 0.70, 0.75, 0.80, 0.90)],
    "PCA":               [{"pca_var":  v} for v in (0.80, 0.90, 0.95)],
}

X_full       = model_df[ACOUSTIC_FULL]                 # all 88 eGeMAPS features
apriori_cols = collapse_functionals(ACOUSTIC_FULL)     # -> 72, by name only
apriori_idx  = [X_full.columns.get_loc(c) for c in apriori_cols]


class ColumnSubset(BaseEstimator, TransformerMixin):
    """Fixed column subset. No fitting -> cannot leak."""
    def __init__(self, idx): self.idx = idx
    def fit(self, X, y=None): return self
    def transform(self, X): return np.asarray(X, dtype=float)[:, self.idx]


class CorrelationClusterSelector(BaseEstimator, TransformerMixin):
    """Spearman |rho| -> average-linkage clustering -> one medoid per cluster.
    Never sees y, but is data-dependent, so fit on training rows only."""
    def __init__(self, threshold=0.75): self.threshold = threshold
    def fit(self, X, y=None):
        corr = np.nan_to_num(spearmanr(np.asarray(X, float)).correlation, nan=0.0)
        corr = (corr + corr.T) / 2
        np.fill_diagonal(corr, 1.0)
        cl = fcluster(linkage(squareform(1 - np.abs(corr), checks=False), "average"),
                      t=1 - self.threshold, criterion="distance")
        absc = np.abs(corr)
        self.support_ = np.sort([i[np.argmax(absc[np.ix_(i, i)].sum(1))]
                                 for i in (np.where(cl == c)[0] for c in np.unique(cl))])
        return self
    def transform(self, X): return np.asarray(X, float)[:, self.support_]


def make_pipe(family, params):
    """Fresh pipeline for one grid point."""
    rf = RandomForestRegressor(random_state=SEED)
    if family == "A-priori collapse":
        return Pipeline([("reduce", ColumnSubset(apriori_idx)), ("rf", rf)])
    if family == "Corr. clustering":
        return Pipeline([("reduce", CorrelationClusterSelector(params["threshold"])), ("rf", rf)])
    return Pipeline([("scale",  StandardScaler()),
                     ("reduce", PCA(n_components=params["pca_var"], random_state=SEED)),
                     ("rf",     rf)])

def label(family, params):
    return family if not params else f"{family} ({next(iter(params.values()))})"

def n_kept(pipe):
    step = pipe.named_steps["reduce"]
    if isinstance(step, ColumnSubset):               return len(step.idx)
    if isinstance(step, CorrelationClusterSelector): return len(step.support_)
    return step.n_components_                        # PCA

def inner_cv(est, X, y, seeds=INNER_SEEDS, n_splits=10):
    rows = []
    for s in seeds:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=s)
        p = cross_val_predict(est, X, y, cv=kf, n_jobs=N_JOBS)
        rows.append({"MAE": mean_absolute_error(y, p),
                     "RMSE": np.sqrt(mean_squared_error(y, p)),
                     "R2": r2_score(y, p)})
    d = pd.DataFrame(rows)
    return {k: (d[k].mean(), d[k].std()) for k in ["MAE", "RMSE", "R2"]}


CONFIGS = [(fam, p) for fam, plist in GRID.items() for p in plist]
print(f"Grid: {len(CONFIGS)} configs x {N_OUTER} outer folds x {len(INNER_SEEDS)} inner seeds")

outer = KFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)
oof        = {m: np.full(len(y), np.nan) for m in SELECTION_METRICS}
fold_rows, inner_rows = [], []

for f, (tr, te) in enumerate(outer.split(X_full), start=1):
    X_tr, X_te = X_full.iloc[tr], X_full.iloc[te]
    y_tr, y_te = y.iloc[tr], y.iloc[te]

    # --- inner: score every grid point on outer_train ONLY (computed once) ---
    scores = {}
    for i, (fam, params) in enumerate(CONFIGS):
        scores[i] = inner_cv(make_pipe(fam, params), X_tr, y_tr)
        inner_rows.append({"fold": f, "config": label(fam, params), "family": fam,
                           "inner_MAE": scores[i]["MAE"][0], "inner_R2": scores[i]["R2"][0]})

    # --- select + evaluate under each metric, from the same inner scores ---
    for metric in SELECTION_METRICS:
        pick = max if HIGHER_IS_BETTER[metric] else min
        i    = pick(scores, key=lambda k: scores[k][metric][0])
        fam, params = CONFIGS[i]

        best = make_pipe(fam, params)
        best.fit(X_tr, y_tr)
        pred = best.predict(X_te)
        oof[metric][te] = pred

        fold_rows.append({"fold": f, "select_on": metric, "winner": label(fam, params),
                          "family": fam, "n_features": n_kept(best), "n_test": len(te),
                          "outer_MAE": mean_absolute_error(y_te, pred),
                          "outer_R2":  r2_score(y_te, pred)})
        print(f"fold {f} [select on {metric:3s}]: {label(fam, params):26s} "
              f"({n_kept(best):>2d} feats)  outer MAE {fold_rows[-1]['outer_MAE']:.3f}")

folds = pd.DataFrame(fold_rows)
inner = pd.DataFrame(inner_rows)

print("\n=== Inner scores per config (mean R2 across outer folds) ===")
print(inner.groupby("config")[["inner_R2", "inner_MAE"]].mean().round(3).to_string())

print("\n=== Outer (unbiased) performance ===")
for metric in SELECTION_METRICS:
    sub, p = folds[folds["select_on"] == metric], oof[metric]
    print(f"\n  selected on {metric}:")
    print(f"    winners            : {sub['winner'].tolist()}")
    print(f"    MAE averaged/folds : {sub['outer_MAE'].mean():.3f} ± {sub['outer_MAE'].std():.3f}")
    print(f"    pooled             : MAE {mean_absolute_error(y, p):.3f}   "
          f"R2 {r2_score(y, p):.3f}   RMSE {np.sqrt(mean_squared_error(y, p)):.3f}   "
          f"r {pearsonr(y, p)[0]:.3f}")

folds.to_csv(DATA_DIR / "nested_cv_folds.csv", index=False)
inner.to_csv(DATA_DIR / "nested_cv_inner_scores.csv", index=False)


Grid: 9 configs x 5 outer folds x 5 inner seeds
fold 1 [select on MAE]: A-priori collapse          (72 feats)  outer MAE 6.461
fold 1 [select on R2 ]: A-priori collapse          (72 feats)  outer MAE 6.461
fold 2 [select on MAE]: A-priori collapse          (72 feats)  outer MAE 6.363
fold 2 [select on R2 ]: A-priori collapse          (72 feats)  outer MAE 6.363
fold 3 [select on MAE]: A-priori collapse          (72 feats)  outer MAE 5.645
fold 3 [select on R2 ]: A-priori collapse          (72 feats)  outer MAE 5.645
fold 4 [select on MAE]: A-priori collapse          (72 feats)  outer MAE 6.404
fold 4 [select on R2 ]: A-priori collapse          (72 feats)  outer MAE 6.404
fold 5 [select on MAE]: A-priori collapse          (72 feats)  outer MAE 6.311
fold 5 [select on R2 ]: A-priori collapse          (72 feats)  outer MAE 6.311

=== Inner scores per config (mean R2 across outer folds) ===
                         inner_R2  inner_MAE
config                                      
A-priori c

In [13]:
print(folds[["fold", "n_test"]])

   fold  n_test
0     1      11
1     2      11
2     3      10
3     4      10
4     5      10
